# Protein Loop Closure as Robot Inverse Kinematics - Student Notebook

This notebook lets you **play with** the loop-closure problem: you know a loop's sequence and the positions of the two residues flanking it, and you want backbone conformations that connect them.

The backbone is a **serial kinematic chain** (phi/psi are the joint angles), so this is the robotics **inverse kinematics** problem. We solve it with **Cyclic Coordinate Descent (CCD)**, then rank the many possible closures by **energy**.

Companion reading: `Kinematics_loop_tutorial.md` (or the Chinese version).

**Prerequisites:** `numpy`, `matplotlib`. (No RDKit needed for this module.)

Run the cells top to bottom, then try the exercises at the end.

## 0. Setup

Run this notebook from the `kinematics_loop/` directory so that the `core` package imports.

In [ ]:
import os, sys

# Make sure the directory containing `core/` is importable.
if not os.path.isdir('core'):
    if os.path.isdir('kinematics_loop/core'):
        os.chdir('kinematics_loop')          # started from the repo root
    else:
        raise RuntimeError('Run this notebook from the kinematics_loop/ directory.')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt

from core import (
    LoopBackbone, LoopProblem, close_loop, optimal_angle,
    backbone_energy, vdw_energy, rama_energy,
    backbone_clashes, rama_violations, normalize, rmsd,
)
print('imports OK, cwd =', os.getcwd())

## 1. Forward kinematics: torsions -> 3D coordinates

Bond lengths and angles barely vary in a peptide, and the peptide bond is planar (omega = 180 deg). So the **only** degrees of freedom are phi and psi per residue - the chain's 'joint angles'.

`LoopBackbone.from_torsions` walks down the chain placing each atom from the previous three (the NeRF construction). Atom order is:

```
C0, N1, CA1, C1, N2, CA2, C2, ..., N_L, CA_L, C_L
```

Note `from_torsions` takes **radians**, while `torsions()` reports **degrees**.

In [ ]:
sequence = 'GSDGKTPN'          # an 8-residue loop
L = len(sequence)

# Two classic backbone conformations, built purely from torsion angles.
helix = LoopBackbone.from_torsions(sequence, np.radians([-57.0]*L), np.radians([-47.0]*L))
extended = LoopBackbone.from_torsions(sequence, np.radians([-135.0]*L), np.radians([135.0]*L))

print('residues       :', helix.n_res)
print('atoms in trace :', len(helix.coords), '= 3*L + 1')
print('rotatable axes :', len(helix.rotatable_axes()), '= 2L-1  (psi of the last residue moves nothing)')

def end_to_end(bb):
    """Distance from the first CA to the last CA."""
    return np.linalg.norm(bb.coords[bb.idx_CA(bb.n_res)] - bb.coords[bb.idx_CA(1)])

print(f'\nhelix    CA1->CA{L} span: {end_to_end(helix):5.2f} A')
print(f'extended CA1->CA{L} span: {end_to_end(extended):5.2f} A')

In [ ]:
def plot_backbone(ax, bb, label=None, **kw):
    """Draw the N-CA-C trace as a 3D line."""
    xyz = bb.coords
    ax.plot(xyz[:, 0], xyz[:, 1], xyz[:, 2], label=label, **kw)

fig = plt.figure(figsize=(11, 4.5))
for k, (bb, name) in enumerate([(helix, 'alpha helix (-57, -47)'),
                                (extended, 'extended (-135, 135)')], 1):
    ax = fig.add_subplot(1, 2, k, projection='3d')
    plot_backbone(ax, bb, marker='o', ms=3, lw=1.5)
    ax.set_title(name)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
plt.tight_layout(); plt.show()

print('Same sequence, same bond lengths and angles - only the torsions differ.')

## 2. Define a closure problem

A loop-closure problem is: a **sequence**, an **N-anchor** (the fixed base atoms `C0, N1, CA1`) and a **C-anchor** (the target atoms `N_L, CA_L, C_L` the loop must reach).

So the notebook is self-checking, we *synthesize* the anchors from a known reference loop with `from_reference`. Then we throw the torsions away and try to rediscover closures. In a real task the anchors instead come from the residues flanking a loop in a PDB structure.

In [ ]:
rng = np.random.default_rng(7)
ref_phi = rng.uniform(-120, -40, size=L)     # from_reference takes DEGREES
ref_psi = rng.uniform(-60, 150, size=L)

problem, reference = LoopProblem.from_reference(sequence, ref_phi, ref_psi)

print('N-anchor (C0, N1, CA1):');    print(np.round(problem.seed, 2))
print('C-anchor (N_L, CA_L, C_L):'); print(np.round(problem.targets, 2))

span = np.linalg.norm(problem.targets[1] - problem.seed[2])
print(f'\nrequired anchor CA-CA span : {span:5.2f} A')
print(f'fully extended span        : {end_to_end(extended):5.2f} A  <- the loop must be able to stretch this far')

## 3. The analytic CCD step - and proof that it is optimal

CCD's trick: for a single torsion, the angle that brings the end-effector closest to its target has a **closed form** - no line search. Rotating by theta changes the objective only through `b*cos(theta) + c*sin(theta)`, so the optimum is

$$ \theta^* = \operatorname{atan2}(c,\, b) $$

Let's verify that empirically: brute-force scan theta over a full turn and check the minimum lands exactly where `optimal_angle` predicts.

In [ ]:
test = LoopBackbone.from_torsions(sequence, np.radians([-135.0]*L), np.radians([135.0]*L), problem.seed)

kind, i = test.rotatable_axes()[0]          # the first torsion (phi of residue 1)
a, b = test.axis_atoms(kind, i)
origin = test.coords[b]
axis = normalize(test.coords[b] - test.coords[a])

theta_star = optimal_angle(test.end_effector(), problem.targets, origin, axis)

# Brute force: rotate by every angle and measure the end-effector RMSD.
thetas = np.linspace(-np.pi, np.pi, 721)
dists = []
for t in thetas:
    trial = test.copy()                     # rotate a throwaway copy
    trial.apply_rotation(kind, i, t)
    dists.append(rmsd(trial.end_effector(), problem.targets))
dists = np.array(dists)

print(f'analytic  theta* = {np.degrees(theta_star):+8.3f} deg')
print(f'brute-force min  = {np.degrees(thetas[dists.argmin()]):+8.3f} deg   (721-point scan)')

plt.figure(figsize=(7, 4))
plt.plot(np.degrees(thetas), dists, lw=1.5, label='end-effector RMSD')
plt.axvline(np.degrees(theta_star), color='crimson', ls='--', label=r'analytic $\theta^*$ = atan2(c, b)')
plt.xlabel(r'rotation about this torsion, $\theta$ (deg)'); plt.ylabel('RMSD to target (A)')
plt.title('one CCD step: the closed form finds the minimum')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. Close the loop

Now sweep that analytic step over **every** torsion, repeatedly, until the end-effector reaches the anchor. That is the whole CCD algorithm.

In [ ]:
start = LoopBackbone.from_torsions(sequence, np.radians([-135.0]*L), np.radians([135.0]*L), problem.seed)
before = start.copy()                      # keep the open starting conformation for plotting

result = close_loop(start, problem.targets)   # modifies `start` IN PLACE
print(f'start RMSD : {rmsd(before.end_effector(), problem.targets):.2f} A')
print(f'converged  : {result.converged}')
print(f'sweeps     : {result.iterations}')
print(f'final RMSD : {result.rmsd:.4f} A')

plt.figure(figsize=(7, 4))
plt.semilogy(np.arange(1, len(result.history) + 1), result.history, lw=1.5)
plt.axhline(0.08, color='crimson', ls='--', label='tolerance = 0.08 A')
plt.xlabel('CCD sweep'); plt.ylabel('end-effector RMSD (A, log scale)')
plt.title('CCD convergence'); plt.legend(); plt.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection='3d')
plot_backbone(ax, before, label='start (open)', color='0.7', lw=1.2)
plot_backbone(ax, start, label='CCD closed', color='tab:blue', lw=2)
plot_backbone(ax, reference, label='reference', color='tab:green', lw=1.2, ls='--')
ax.scatter(*problem.targets.T, color='crimson', s=60, label='C-anchor (target)')
ax.scatter(*problem.seed.T, color='black', s=60, label='N-anchor (fixed)')
ax.set_title('CCD drives the chain end onto the anchor')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

print('The closed loop reaches the same ENDS as the reference,')
print('but its middle need not follow the reference path at all.')

## 5. Closure is under-determined

A loop has `2L-1` adjustable torsions but only needs to satisfy 6 constraints (position + orientation of one end). So there are **infinitely many** closed conformations, and CCD finds whichever one happens to be nearest its starting point.

Let's see it: close the loop from several different random starts and overlay the results.

In [ ]:
closures = []
rng2 = np.random.default_rng(0)
while len(closures) < 4:
    bb = LoopBackbone.from_torsions(
        sequence, np.radians(rng2.uniform(-180, 180, L)),
        np.radians(rng2.uniform(-180, 180, L)), problem.seed)
    if close_loop(bb, problem.targets).converged:
        closures.append(bb)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection='3d')
for k, bb in enumerate(closures, 1):
    plot_backbone(ax, bb, label=f'closure {k}  (E={backbone_energy(bb):.1f})', lw=1.6)
ax.scatter(*problem.targets.T, color='crimson', s=60, label='C-anchor')
ax.scatter(*problem.seed.T, color='black', s=60, label='N-anchor')
ax.set_title('four DIFFERENT loops, all closing the same two anchors')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

for k, bb in enumerate(closures, 1):
    print(f'closure {k}: end-effector RMSD = {rmsd(bb.end_effector(), problem.targets):.4f} A')
print('\nAll below the 0.08 A tolerance - so closure geometry alone cannot rank them.')
print('That is why we need an energy.')

## 6. Ranking closures by energy

CCD only makes the ends *meet*. To decide which closure is **good** we score each with a coarse backbone energy:

$$ E = E_{vdW} + w_{rama}\, E_{rama} $$

- `vdw_energy` - Lennard-Jones 12-6 over non-adjacent backbone atoms, so self-intersecting loops cost a lot.
- `rama_energy` - a smooth penalty growing with each residue's distance from the nearest favored (phi, psi) basin.

`LoopProblem.solve` samples a **pool** of closures, scores them all, and returns the lowest-energy ones.

Two things to watch for when you run the cells below:

1. At `w_rama=1` the **vdW term dominates**: `E_vdw` reaches the hundreds while `E_rama` stays around 10-40. So the ranking mostly buys you *clash-free* loops (energy correlates ~0.9 with clash count) and only weakly improves Ramachandran quality. Exercise E2 lets you rebalance the two.
2. This energy sees only the N-CA-C trace - no side chains, no hydrogen bonds, no solvent. It is a **ranking** score for comparing closures of the same loop, *not* a stability estimate.

In [ ]:
print('the two terms, for the four closures above:')
for k, bb in enumerate(closures, 1):
    phi, psi = bb.torsions()
    print(f'  closure {k}:  E_vdw={vdw_energy(bb.coords):8.2f}   '
          f'E_rama={rama_energy(phi, psi):6.2f}   '
          f'total={backbone_energy(bb):8.2f}   '
          f'clashes={backbone_clashes(bb)}  rama_bad={rama_violations(phi, psi)}')

In [ ]:
solutions = problem.solve(n_solutions=5, max_tries=300, seed=1)

print(f"{'#':>2} {'energy':>9} {'rmsd':>7} {'sweeps':>7} {'clashes':>8} {'rama_bad':>9}")
for k, s in enumerate(solutions, 1):
    print(f'{k:>2} {s.energy:9.2f} {s.rmsd:7.3f} {s.iterations:7d} {s.clashes:8d} {s.rama_bad:9d}')

print('\nEvery row closes equally well (rmsd ~ 0.08), so ENERGY is what ranks them.')

In [ ]:
# Does the energy actually track structural quality? Score a bigger pool and look.
# (This runs ~30 CCD closures, so give it a few seconds.)
pool = problem.solve(n_solutions=30, max_tries=300, candidate_pool=30, seed=5)
energies = np.array([s.energy for s in pool])
rama_bad = np.array([s.rama_bad for s in pool])
clashes  = np.array([s.clashes for s in pool])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(energies, bins=12, color='tab:blue', alpha=0.8)
axes[0].set_xlabel('backbone energy'); axes[0].set_ylabel('count')
axes[0].set_title(f'energies of {len(pool)} closures')

jitter = np.random.default_rng(1).uniform(-0.15, 0.15, len(pool))
sc = axes[1].scatter(energies, rama_bad + jitter, c=clashes, cmap='coolwarm', s=40)
axes[1].set_xlabel('backbone energy'); axes[1].set_ylabel('Ramachandran outliers')
axes[1].set_title('colour = number of clashes')
plt.colorbar(sc, ax=axes[1], label='clashes')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'energy range: {energies.min():.1f} to {energies.max():.1f}')
print(f'correlation(energy, rama_bad) = {np.corrcoef(energies, rama_bad)[0, 1]:+.2f}')
print(f'correlation(energy, clashes)  = {np.corrcoef(energies, clashes)[0, 1]:+.2f}' 
      if clashes.std() > 0 else 'no clashes in this pool, so that term is flat')

## 7. Export for PyMOL / VMD

Write the best conformation and the reference as PDB files you can open in a viewer.

In [ ]:
os.makedirs('examples/output', exist_ok=True)
reference.write_pdb('examples/output/nb_reference.pdb')
solutions[0].backbone.write_pdb('examples/output/nb_best.pdb')
print('wrote nb_reference.pdb and nb_best.pdb to examples/output/')

phi, psi = solutions[0].backbone.torsions()
print('\nbest conformation torsions (deg):')
print('res  ' + ' '.join(f'{c:>7}' for c in sequence))
print('phi  ' + ' '.join(f'{v:7.1f}' for v in phi))
print('psi  ' + ' '.join(f'{v:7.1f}' for v in psi))
print('\n(psi of the last residue is reported as 0 - it is not part of the trace.)')

## Exercises

Fill in the `...` and run. Compare your results with a neighbour!

**E1. Does the starting conformation matter?** Close the loop from the helix, the extended chain, and a random start. Compare the number of sweeps and the final energy.

In [ ]:
starts = {
    'helix':    (np.full(L, -57.0),  np.full(L, -47.0)),
    'extended': (np.full(L, -135.0), np.full(L, 135.0)),
    'random':   (rng.uniform(-180, 180, L), rng.uniform(-180, 180, L)),
}
for name, (p0, s0) in starts.items():
    bb = LoopBackbone.from_torsions(sequence, np.radians(p0), np.radians(s0), problem.seed)
    res = close_loop(bb, problem.targets)
    # TODO: print name, res.converged, res.iterations and backbone_energy(bb)
    ...

**E2. Crank the Ramachandran weight.** Re-solve with `w_rama` = 0, 1 and 10. Does the top solution's `rama_bad` count drop when you weight it more heavily? What happens to `clashes`?

In [ ]:
for w in (0.0, 1.0, 10.0):
    best = problem.solve(n_solutions=3, max_tries=200, w_rama=w, seed=2)[0]
    # TODO: print w, best.energy, best.rama_bad, best.clashes
    ...

**E3. How big does the candidate pool need to be?** The pool is the whole point of multi-start: score more closures, find a better one. Solve with `candidate_pool` = 5, 15, 30 and record the *best* energy found each time. Where do the returns start to diminish?

In [ ]:
for p in (5, 15, 30):
    # TODO: solve with candidate_pool=p (use max_tries=300, seed=3),
    #       then print p and the lowest energy returned.
    ...

**E4. Loop length.** Repeat the multi-start solve for a 4-residue and a 12-residue loop. Which is harder to close? Which gives a wider spread of energies, and why?

In [ ]:
for seq in ('GSDG', 'GSDGKTPNLAEV'):
    n = len(seq)
    r = np.random.default_rng(11)
    prob, _ = LoopProblem.from_reference(seq, r.uniform(-120, -40, n), r.uniform(-60, 150, n))
    # TODO: solve `prob` and print the sequence length, how many solutions came
    #       back, and the best energy.
    ...

---

### Where to go next

- **Rama-biased restarts:** draw the random starting torsions from the favored basins instead of uniformly over the circle. How many fewer tries do you need for a good closure?
- **Analytic KIC:** replace CCD with exact Kinematic Closure, which solves the final three torsions in closed form (Coutsias et al. 2004) rather than iterating.
- **Real anchors:** instead of `from_reference`, read the two flanking residues out of a PDB file and rebuild a genuinely missing loop.

See section 10 of `Kinematics_loop_tutorial.md` for more.